# AG-CGCNN · Training and prediction

A reproducible introduction to the released implementation. Start with the explicitly **synthetic software demonstration** below, then replace the dataset configuration with real MOF inputs. These demo targets have no physical meaning and cannot reproduce the paper.

Install the repository with `python -m pip install -e ".[notebooks,explain]"` before opening Jupyter.

## 1. Locate the repository

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "agcgcnn").exists():
    ROOT = ROOT.parent
assert (ROOT / "agcgcnn").exists(), "Open this notebook from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
import torch
torch.set_num_threads(1)


## 2. Prepare a small synthetic dataset

Each CSV row contains an identifier, five descriptors, and one regression target. The CIF archive and atom embeddings sit beside the CSV. For real experiments use the descriptor order and units in `docs/DATA.md` and the 92-dimensional CGCNN atom embeddings.

Existing run directories are not overwritten. Change `RUN_NAME` when rerunning.

In [ ]:
from scripts.make_demo_data import make_demo
RUN_NAME = "notebook-demo"
DATA = ROOT / "runs" / RUN_NAME / "data"
TRAIN = ROOT / "runs" / RUN_NAME / "training"
PREDICT = ROOT / "runs" / RUN_NAME / "predictions"
make_demo(DATA, count=30)


## 3. Train AG-CGCNN

The model encodes atoms and bonds, pools the crystal representation, appends standardized geometric descriptors, and predicts the target. Splits use a fixed seed; scalers are fitted **only to the training split**. Validation loss selects the checkpoint; the test split is evaluated afterward.

The small hidden dimensions below are for the demonstration. The release defaults use 64 atom features, 128 hidden units and three convolutions.

In [ ]:
from agcgcnn.train import parser, run
args = parser().parse_args([
    "--data", str(DATA), "--output", str(TRAIN),
    "--epochs", "2", "--batch-size", "8", "--atom-fea-len", "8",
    "--hidden", "12", "--convolutions", "1", "--seed", "123"
])
metrics = run(args)
metrics


## 4. Inspect learning history

The metric is MAE in target units for regression, or accuracy for classification. The synthetic test metric verifies software behavior only.

In [ ]:
import pandas as pd
history = pd.read_csv(TRAIN / "history.csv")
history.plot(x="epoch", y=["train_loss", "validation_loss"], marker="o", title="Synthetic demonstration: loss")


## 5. Predict and export the pooled representation

`encoded.npy` contains the pooled graph embedding followed by standardized geometric descriptors. The explainability notebook reuses this representation and the same trained prediction head.

In [ ]:
from agcgcnn.predict import run as predict
predict(TRAIN / "best.pt", DATA, PREDICT)
pd.read_csv(PREDICT / "predictions.csv").head()


## 6. Move to a research dataset

Replace `DATA` with your data directory and use a new output directory. Set `--task classification --classes 4` for four-class prediction; set `--targets 2` for two regression outputs. Classes must be labeled consistently before training. Do not infer published class definitions from the synthetic data.

See [usage](../docs/USAGE.md), [data conventions](../docs/DATA.md), and [research provenance](../docs/PROVENANCE.md).